# Continue fine-tuning the SQuAD-trained BERT model on NewsQA


This takes the BERT model already fine-tuned on SQuAD v1.1 (`qa_model` in Drive, from `train_qa.ipynb`) and continues fine-tuning it on the NewsQA subset, instead of training a fresh model from scratch or mixing the two datasets together.

Requires `train-newsqa.json` and `dev-newsqa.json` to already exist in the `qa/` folder on Drive. If they don't, run `preprocess_newsqa.ipynb` first.

The resulting model is saved to a new folder, `qa_model_newsqa`, so the original SQuAD-only checkpoint in `qa_model` is left untouched for comparison.

This also re-scores the new model on the original SQuAD v1.1 dev set at the end, to check whether fine-tuning on NewsQA hurt SQuAD performance (a known risk with continued fine-tuning, sometimes called catastrophic forgetting).

In [1]:
!pip install -q -U transformers datasets accelerate evaluate

import transformers
print("transformers version:", transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
transformers version: 5.20.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import collections
import subprocess
import dataclasses
import inspect

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")

# Starting point: the SQuAD-trained model from train_qa.ipynb, not a fresh checkpoint.
BASE_MODEL_DIR = os.path.join(PROJECT_DIR, "qa_model")

NEWSQA_TRAIN_FILE = os.path.join(DATA_DIR, "train-newsqa.json")
NEWSQA_DEV_FILE = os.path.join(DATA_DIR, "dev-newsqa.json")
SQUAD_DEV_FILE = os.path.join(DATA_DIR, "dev-v1.1.json")
EVAL_SCRIPT = os.path.join(DATA_DIR, "evaluate-v2.0.py")

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2
# Lower than the 3e-5 used for the original SQuAD fine-tune: the model is
# already converged, so a smaller LR for continued fine-tuning reduces the
# risk of wrecking what it already learned on SQuAD.
LEARNING_RATE = 2e-5
NUM_EPOCHS = 2
N_BEST = 20
MAX_ANSWER_LENGTH = 30

# New output directory. Deliberately NOT overwriting qa_model, so the
# original SQuAD-only baseline stays available for comparison.
OUTPUT_DIR = os.path.join(PROJECT_DIR, "qa_model_newsqa")
os.makedirs(OUTPUT_DIR, exist_ok=True)

PREDICTIONS_NEWSQA_FILE = os.path.join(DATA_DIR, "predictions_newsqa_finetuned.json")
PREDICTIONS_SQUAD_RECHECK_FILE = os.path.join(DATA_DIR, "predictions_squad_after_newsqa_finetune.json")

MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

In [4]:
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > Hardware "
    "accelerator > GPU (T4), save, reconnect, then re-run from the top."
)
print("GPU:", torch.cuda.get_device_name(0))

for path in (BASE_MODEL_DIR, NEWSQA_TRAIN_FILE, NEWSQA_DEV_FILE, SQUAD_DEV_FILE, EVAL_SCRIPT):
    assert os.path.exists(path), (
        f"Missing: {path}\n"
        "If this is train-newsqa.json / dev-newsqa.json, run preprocess_newsqa.ipynb first.\n"
        "If this is qa_model, make sure train_qa.ipynb has been run and saved there."
    )
print("All required files found.")

GPU: Tesla T4
All required files found.


In [5]:
def load_squad_json(path):
    with open(path) as f:
        squad = json.load(f)
    ids, questions, contexts, answers = [], [], [], []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                ids.append(qa["id"])
                questions.append(qa["question"])
                contexts.append(context)
                answers.append(
                    {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    }
                )
    return Dataset.from_dict(
        {"id": ids, "question": questions, "context": contexts, "answers": answers}
    )


print("Loading NewsQA data...")
raw_datasets = DatasetDict(
    {
        "train": load_squad_json(NEWSQA_TRAIN_FILE),
        "validation": load_squad_json(NEWSQA_DEV_FILE),
    }
)

print("Loading original SQuAD dev set, for the forgetting check at the end...")
squad_dev_raw = load_squad_json(SQUAD_DEV_FILE)

if MAX_TRAIN_SAMPLES is not None:
    raw_datasets["train"] = raw_datasets["train"].select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES is not None:
    raw_datasets["validation"] = raw_datasets["validation"].select(range(MAX_EVAL_SAMPLES))

raw_datasets

Loading NewsQA data...
Loading original SQuAD dev set, for the forgetting check at the end...


DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 10003
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 1003
    })
})

In [6]:
# Load the tokenizer from the already fine-tuned model's own saved files,
# not from "bert-base-uncased", so it matches exactly what qa_model was saved with.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
assert tokenizer.is_fast, "Need a fast tokenizer for offset mapping"

In [7]:
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    tokenized["example_id"] = example_ids
    return tokenized


print("Tokenizing NewsQA train set...")
train_dataset = raw_datasets["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

print("Tokenizing NewsQA validation set...")
validation_dataset = raw_datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

print("Tokenizing original SQuAD dev set (for the forgetting check)...")
squad_validation_dataset = squad_dev_raw.map(
    prepare_validation_features,
    batched=True,
    remove_columns=squad_dev_raw.column_names,
)

Tokenizing NewsQA train set...


Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Tokenizing NewsQA validation set...


Map:   0%|          | 0/1003 [00:00<?, ? examples/s]

Tokenizing original SQuAD dev set (for the forgetting check)...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [8]:
# Load the SQuAD-trained weights as the starting point, not a fresh bert-base-uncased.
model = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL_DIR)

ta_fields = {f.name for f in dataclasses.fields(TrainingArguments)}


def pick(*name_value_pairs):
    for name, value in name_value_pairs:
        if name in ta_fields:
            return {name: value}
    return {}


ta_kwargs = {"output_dir": OUTPUT_DIR}
ta_kwargs.update(pick(("eval_strategy", "no"), ("evaluation_strategy", "no")))
ta_kwargs.update(pick(("save_strategy", "epoch")))
ta_kwargs["learning_rate"] = LEARNING_RATE
ta_kwargs["per_device_train_batch_size"] = BATCH_SIZE
ta_kwargs["per_device_eval_batch_size"] = BATCH_SIZE
ta_kwargs["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
ta_kwargs["num_train_epochs"] = NUM_EPOCHS
ta_kwargs["weight_decay"] = 0.01
ta_kwargs.update(pick(("warmup_ratio", 0.1)))
ta_kwargs.update(pick(("fp16", torch.cuda.is_available())))
ta_kwargs.update(pick(("gradient_checkpointing", True)))
ta_kwargs.update(pick(("dataloader_num_workers", 0)))
ta_kwargs.update(pick(("report_to", "none")))
ta_kwargs.update(pick(("run_name", "bert-newsqa-continued-finetune")))

args = TrainingArguments(**ta_kwargs)

trainer_params = set(inspect.signature(Trainer.__init__).parameters)
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    **{tokenizer_kwarg: tokenizer},
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
print("Fine-tuning on NewsQA...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Fine-tuning on NewsQA...


Step,Training Loss
500,1.283624
1000,1.125547
1500,1.038970
2000,0.749085
2500,0.700418
3000,0.704985


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/CS4248 Group Project/qa_model_newsqa


## Evaluate on NewsQA dev

Same inference + decoding approach as `train_qa.ipynb`.

In [10]:
def run_inference(model, dataset, batch_size=BATCH_SIZE):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    eval_set_for_model = dataset.remove_columns(["example_id", "offset_mapping"])
    eval_set_for_model.set_format("torch")

    all_start_logits, all_end_logits = [], []
    with torch.no_grad():
        for i in range(0, len(eval_set_for_model), batch_size):
            batch = eval_set_for_model[i : i + batch_size]
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            all_start_logits.append(outputs.start_logits.cpu().numpy())
            all_end_logits.append(outputs.end_logits.cpu().numpy())

    return np.concatenate(all_start_logits, axis=0), np.concatenate(all_end_logits, axis=0)


def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits):
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_id]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        context = example["context"]
        feature_indices = features_per_example[example_index]

        valid_answers = []
        for feature_index in feature_indices:
            start_logit = all_start_logits[feature_index]
            end_logit = all_end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offsets)
                        or end_index >= len(offsets)
                        or offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                        continue
                    start_char, end_char = offsets[start_index][0], offsets[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logit[start_index] + end_logit[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if valid_answers:
            best = max(valid_answers, key=lambda x: x["score"])
            predictions[example["id"]] = best["text"]
        else:
            predictions[example["id"]] = ""
    return predictions


def score(data_file, predictions_file):
    result = subprocess.run(
        ["python", EVAL_SCRIPT, data_file, predictions_file],
        check=True, capture_output=True, text=True,
    )
    print(result.stdout)

In [11]:
print("Running inference on NewsQA validation set...")
newsqa_start_logits, newsqa_end_logits = run_inference(model, validation_dataset)

print("Decoding NewsQA answer spans...")
newsqa_predictions = postprocess_qa_predictions(
    raw_datasets["validation"], validation_dataset, newsqa_start_logits, newsqa_end_logits
)

with open(PREDICTIONS_NEWSQA_FILE, "w") as f:
    json.dump(newsqa_predictions, f)
print(f"Predictions written to {PREDICTIONS_NEWSQA_FILE}")

print("Scoring on NewsQA dev...")
score(NEWSQA_DEV_FILE, PREDICTIONS_NEWSQA_FILE)

Running inference on NewsQA validation set...
Decoding NewsQA answer spans...
Predictions written to /content/drive/MyDrive/CS4248 Group Project/qa/predictions_newsqa_finetuned.json
Scoring on NewsQA dev...
{
  "exact": 48.554336989032905,
  "f1": 64.23558372603487,
  "total": 1003,
  "HasAns_exact": 48.554336989032905,
  "HasAns_f1": 64.23558372603487,
  "HasAns_total": 1003
}



## Forgetting check: re-score on the original SQuAD dev set

Compares against the original baseline (EM 80.92 / F1 88.40, recorded in `bert_baseline_results.md`). If these numbers drop sharply after fine-tuning on NewsQA, that is catastrophic forgetting and worth flagging in the report.

In [12]:
print("Running inference on original SQuAD dev set...")
squad_start_logits, squad_end_logits = run_inference(model, squad_validation_dataset)

print("Decoding SQuAD answer spans...")
squad_predictions = postprocess_qa_predictions(
    squad_dev_raw, squad_validation_dataset, squad_start_logits, squad_end_logits
)

with open(PREDICTIONS_SQUAD_RECHECK_FILE, "w") as f:
    json.dump(squad_predictions, f)
print(f"Predictions written to {PREDICTIONS_SQUAD_RECHECK_FILE}")

print("Scoring on original SQuAD dev (post NewsQA fine-tuning)...")
score(SQUAD_DEV_FILE, PREDICTIONS_SQUAD_RECHECK_FILE)

Running inference on original SQuAD dev set...
Decoding SQuAD answer spans...
Predictions written to /content/drive/MyDrive/CS4248 Group Project/qa/predictions_squad_after_newsqa_finetune.json
Scoring on original SQuAD dev (post NewsQA fine-tuning)...
{
  "exact": 74.84389782403028,
  "f1": 85.43684069218847,
  "total": 10570,
  "HasAns_exact": 74.84389782403028,
  "HasAns_f1": 85.43684069218847,
  "HasAns_total": 10570
}



In [13]:
score(NEWSQA_DEV_FILE, PREDICTIONS_NEWSQA_FILE)

{
  "exact": 48.554336989032905,
  "f1": 64.23558372603487,
  "total": 1003,
  "HasAns_exact": 48.554336989032905,
  "HasAns_f1": 64.23558372603487,
  "HasAns_total": 1003
}

